# 2 — 声を覚える

GPT-SoVITS の環境を準備し、録音した声を参照音声として設定します。

この段階では追加学習は行わず、参照音声によるゼロショット音声合成を使います。

**注意:** Colab の同じランタイムで「声を録る」の出力ファイルを利用してください。ランタイムを切り替えるとファイルやインストール環境は引き継がれません。

In [ ]:
import os, sys, subprocess, pathlib, json, time
import torch

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Kaggle Notebook の Accelerator を GPU に設定してから再実行してください。')

WORK = pathlib.Path('/kaggle/working')
REPO = WORK / 'GPT-SoVITS'
OUT = WORK / 'voice_clone_output'
OUT.mkdir(parents=True, exist_ok=True)

## 2. 声を覚える — GPT-SoVITS を準備する

公式 Linux インストーラーを使います。GPU 世代や Kaggle のランタイム構成によっては依存関係の調整が必要になる場合があります。エラーが出たら、ログ末尾を確認してください。

In [ ]:
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/RVC-Boss/GPT-SoVITS.git', str(REPO)], check=True)
else:
    print('既存 clone を使用:', REPO)

subprocess.run(['apt-get', 'update', '-qq'], check=True)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libsndfile1'], check=True)
os.chdir(REPO)

# 公式 README の Linux セットアップに沿って依存関係と事前学習モデルを準備
install_log = OUT / 'install.log'
with install_log.open('w') as log:
    result = subprocess.run(['bash', 'install.sh', '--device', 'CU126', '--source', 'HF'], stdout=log, stderr=subprocess.STDOUT, text=True)
if result.returncode != 0:
    print('セットアップ失敗。ログ末尾:')
    print('\n'.join(install_log.read_text(errors='replace').splitlines()[-80:]))
    raise RuntimeError(f'install.sh failed: {result.returncode}')
print('セットアップ完了。ログ:', install_log)

## 2. 声を覚える — 参照音声を指定

`AUDIO_INPUT` と `TRANSCRIPT_INPUT` を、自分の Kaggle Input のパスに変更してください。文字起こしは録音内容と一致させます。参照音声は自分の声、または利用許可のある声だけを使用してください。

In [ ]:
from pathlib import Path
from IPython.display import Audio, display

# TODO: Kaggle の Input に表示される実際のパスへ変更
AUDIO_INPUT = Path('/kaggle/input/your-voice-data/voice_mono_32k.wav')
TRANSCRIPT_INPUT = Path('/kaggle/input/your-voice-data/reference_transcript.txt')

if not AUDIO_INPUT.is_file():
    raise FileNotFoundError(f'音声ファイルがありません: {AUDIO_INPUT}。AUDIO_INPUT を修正してください。')
if not TRANSCRIPT_INPUT.is_file():
    raise FileNotFoundError(f'文字起こしがありません: {TRANSCRIPT_INPUT}。TRANSCRIPT_INPUT を修正してください。')

prompt_text = TRANSCRIPT_INPUT.read_text(encoding='utf-8').strip()
if not prompt_text:
    raise ValueError('参照音声の文字起こしが空です。')
print('参照音声:', AUDIO_INPUT)
print('参照テキスト:', prompt_text)
display(Audio(filename=str(AUDIO_INPUT)))

# ここを読み上げたい日本語に変更
TARGET_TEXT = 'こんにちは。これは、自分の声を使った日本語音声合成のテストです。'
print('生成する文:', TARGET_TEXT)

## 次へ

参照音声のパスと文字起こしを控えてください。次の Notebook で API を起動して音声を生成します。同じ Colab ランタイムを使い続けてください。